In [ ]:
############################
####  PANEL STEROWANIA   ###
############################

In [37]:
import json, os
from datetime import datetime

control_file = "/home/jovyan/notebooks/Nuclear-Power-Station/control.json"

# Wartości domyślne z producer.py
DEFAULTS = {
    "rods_out":      55.0,
    "boron":         700,
    "fault":         "NONE",
    "safety_system": "NONE"
}

if os.path.exists(control_file):
    with open(control_file, "r") as f:
        current = json.load(f)
else:
    current = {}

rods    = current.get("rods_out",      DEFAULTS["rods_out"])
boron   = current.get("boron",         DEFAULTS["boron"])
fault   = current.get("fault",         DEFAULTS["fault"])
safety  = current.get("safety_system", DEFAULTS["safety_system"])

source = "domyślne" if not os.path.exists(control_file) else "aktualne"

print(f"📊 Ustawienia {source} | {datetime.now().strftime('%H:%M:%S')}")
print(f"   Pręty:           {rods}%")
print(f"   Bor:             {boron} ppm")
print(f"   Usterka:         {fault}")
print(f"   System awaryjny: {safety}")

📊 Ustawienia aktualne | 10:37:06
   Pręty:           55.0%
   Bor:             700 ppm
   Usterka:         NONE
   System awaryjny: NONE


In [32]:
import json
from datetime import datetime

control_file = "/home/jovyan/notebooks/Nuclear-Power-Station/control.json"

rods_out = 75.0  # ← ustaw tylko moc (0-100%)

# Bor wyliczany automatycznie im wyższa moc, tym mniej boru, ponieaż bor hamuje reakcję, więc przy wysokiej mocy powinno go być mało, a dodatkowo trzeba uwzględić flux od prętów
boron = round(rods_out * 20)

if rods_out > 90:
    print(f"⚠️  Moc powyżej 90% — monitoruj temperaturę i promieniowanie!")

with open(control_file, "w") as f:
    json.dump({"rods_out": rods_out, "boron": boron}, f)

print(f"⚡ Zmiana mocy | {datetime.now().strftime('%H:%M:%S')}")
print(f"   Pręty wysunięte: {rods_out}%")
print(f"   Bor (auto):      {boron} ppm")
print(f"   Szacowana moc:   ~{rods_out:.0f}%  (~{rods_out * 10:.0f} MWe)")

⚡ Zmiana mocy | 10:35:24
   Pręty wysunięte: 75.0%
   Bor (auto):      1500 ppm
   Szacowana moc:   ~75%  (~750 MWe)


In [34]:
import json
import os

control_file = "/home/jovyan/notebooks/Nuclear-Power-Station/control.json"

if os.path.exists(control_file):
    with open(control_file, "r") as f:
        state = json.load(f)
    
    fault = state.get("fault", "NONE")
    safety = state.get("safety_system", "NONE")
    rods = state.get("rods_out", "—")
    boron = state.get("boron", "—")
    
    fault_icons = {
        "NONE": "🟢", "LEAK": "💧", 
        "PUMP_FAIL": "⚙️", "TURBINE_TRIP": "⚡", "TOTAL_BLACKOUT": "☢️"
    }
    safety_icons = {
        "NONE": "🟢", "SCRAM": "🔴", 
        "STEAM_DUMP": "💨", "EMERGENCY_BORON": "🧪"
    }
    
    print("=" * 45)
    print("       STAN REAKTORA PWR-UNIT-01")
    print("=" * 45)
    print(f"  Usterka:         {fault_icons.get(fault, '❓')} {fault}")
    print(f"  System awaryjny: {safety_icons.get(safety, '❓')} {safety}")
    if rods != "—":
        print(f"  Pręty wysunięte: {rods}%")
    if boron != "—":
        print(f"  Stężenie boru:   {boron} ppm")
    print("=" * 45)
else:
    print("⚠️ Brak pliku control.json — producer nie był jeszcze sterowany")
    print("   Reaktor działa na parametrach domyślnych (60% mocy)")

       STAN REAKTORA PWR-UNIT-01
  Usterka:         🟢 NONE
  System awaryjny: 🟢 NONE
  Pręty wysunięte: 75.0%
  Stężenie boru:   1500 ppm


In [ ]:
### KONTROLOWANIE REAKTORA ###
# trzeba w odzielnym termianlu wpisać poniższą komendę, ustawiając wartości rods_out i boron;
# rods_out: 0 - reaktor wygaszony, 100 - max moc, boron: tym więcej boru, tym spokojniejsza reakcja
# Kod: echo '{"rods_out": 60.0, "boron": 700}' > control.json
#
###  SYMULOWANIE USTEREK   ###
# aby zasymolować awarię, w odzielnym terminalu trzeba wpisać poniższą komendę, wybierająć rodzaj fault;
# echo '{"fault": "NONE"}' > control.json           - brak usterki
# echo '{"fault": "LEAK"}' > control.json           - Wyciek: Ciśnienie pierwotne spada, ciśnienie w kopule rośnie, promieniowanie rośnie 
# echo '{"fault": "PUMP_FAIL"}' > control.json      - Awaria pompy: Przepływ spada do 20%, temperatura hot szybuje w górę
# echo '{"fault": "TURBINE_TRIP"}' > control.json   - Odłączenie sieci -> Wzrost ciśnienia pary -> Wzrost temp. reaktora
# echo '{"fault": "TOTAL_BLACKOUT"}' > control.json - KATASTROFA: Całkowity brak prądu w elektrowni (SBO) -> Przepływ pierwotny gwałtownie spada do zera
#                                                         -> Układ wtórny całkowicie zamiera (pompy wody zasilającej stoją) -> Ekstremalny wzrost temperatury (brak odbioru ciepła)
###  SYSTEMY AWARYJNE   ###
# echo '{"safety_system": "NONE"}' > control.json             - systemy awaryjne wyłączone
# echo '{"safety_system": "STEAM_DUMP"}' > control.json       - Zrzut pary: gwałtowny spadek ciśnienia i temperatury -> Jeśli mamy wyciek, zrzut pary wyrzuca skażoną parę na zewnątrz
# echo '{"safety_system": "EMERGENCY_BORON"}' > control.json  - Szybkie borowanie (chemiczne hamowanie reakcji)
# echo '{"safety_system": "SCRAM"}' > control.json            - w ostateczności: całkowite zrzucenie prętów i wygasznie reaktora
# Jeśli włączymy system awaryjny to potem musimy go wyłączyć!


In [23]:
import json

# Wywołaj awarię
with open("/home/jovyan/notebooks/Nuclear-Power-Station/control.json", "w") as f:
    json.dump({"fault": "LEAK"}, f)
print("✅ Awaria wycieku aktywowana | {datetime.now().strftime('%H:%M:%S')}")

✅ Awaria wycieku aktywowana | {datetime.now().strftime('%H:%M:%S')}


In [35]:
# Awaria pompy
with open("/home/jovyan/notebooks/Nuclear-Power-Station/control.json", "w") as f:
    json.dump({"fault": "PUMP_FAIL"}, f)
print("✅ Awaria pompy aktywowana | {datetime.now().strftime('%H:%M:%S')}")

✅ Awaria pompy aktywowana | {datetime.now().strftime('%H:%M:%S')}


In [19]:
import json
from datetime import datetime

# Całkowity blackout
with open("/home/jovyan/notebooks/Nuclear-Power-Station/control.json", "w") as f:
    json.dump({"fault": "TOTAL_BLACKOUT"}, f)
print(f"✅ Awaria TOTAL_BLACKOUT aktywowana | {datetime.now().strftime('%H:%M:%S')}")

✅ Awaria TOTAL_BLACKOUT aktywowana | 10:20:27


In [36]:
# Powrót do normy
with open("/home/jovyan/notebooks/Nuclear-Power-Station/control.json", "w") as f:
    json.dump({"fault": "NONE"}, f)
print("✅ Powrót do normy | {datetime.now().strftime('%H:%M:%S')}")

✅ Powrót do normy | {datetime.now().strftime('%H:%M:%S')}
